# decision_lab — tuning notebook

The same read assembly `python -m decision_lab report` and the dashboard's Runs page both call —
`decision_lab.analysis`, plus `registry`, `corpus`, `candidates` and `sweep` beneath it (§14:
one implementation, three front doors). A notebook that re-derived scoring here would be a fourth.

Nothing below spends a cent, reaches a provider, or writes to the workspace. It reads a corpus and
a sweep that a `decision_lab sweep` or `calibrate` run has already written to disk, and re-derives
the same tables the dashboard shows.

In [ ]:
from pathlib import Path

from decision_lab import analysis, candidates, corpus, registry, sweep
from tradebot.core.clock import SystemClock

WORKSPACE = None  # None means decision_lab/workspace; set a Path to read another
registry.read_all(workspace=WORKSPACE)[-10:]

## Pick a run

Copy a `corpus_id` from the rows above. `sweep.latest_meta` finds the most recent sweep that
corpus ran; `analysis.analyse_matrix` folds its candidates' rows onto the corpus and scores them —
the same fold `cli.report` and the dashboard's run detail page use, never a second scorer.

In [ ]:
import asyncio

CORPUS_ID = "8ac130d8f2ed5650dff0dcb9f969d07e"  # any id from the rows above


async def _prepare() -> analysis.MatrixAnalysis:
    loaded = corpus.load(CORPUS_ID, workspace=WORKSPACE)
    result = sweep.latest_meta(CORPUS_ID, workspace=WORKSPACE)
    matrix = candidates.load_matrix(
        Path(result.matrix_source), reference=loaded.meta.reference_basket
    )
    scoring = await analysis.build_scoring(Path(loaded.meta.dataset_directory), clock=SystemClock())
    return analysis.analyse_matrix(
        loaded, matrix, result.matrix_digest, scoring, workspace=WORKSPACE
    )


analysed = asyncio.run(_prepare())
analysed.ranking

## The seat tables

Per candidate, per seat: round 0 beside the final vote (§9.7) — under `blind_then_debate` a
seat's later votes are contaminated by its peers by design, so "reasons well" and "is easily
talked round" are different questions. A candidate that produced no scored decision is listed
beside them with its reason, never dropped and never ranked at 0.0% (finding 3).

In [ ]:
analysed.candidate_seats, analysed.not_measured

In [ ]:
rows = registry.read_all(workspace=WORKSPACE)
left, right = rows[-2], rows[-1]
tuple(
    (name, getattr(left, name), getattr(right, name))
    for name in ("candidate_id", "accuracy", "cost_usd", "net_profit", "matrix_digest")
    if getattr(left, name) != getattr(right, name)
)